# Nosh Mish Mosh: Sample Size for an Artisanal A/B Test — Complete Solution

This notebook contains fully worked answers, alternate implementations, extra practice solutions, a Monte-Carlo verification, and a ready-to-run simulation playground.

Use it to check your work after you finish the **Skeleton** notebook.

---


## Flowchart: Desired Outcome for the Nosh Mish Mosh Experiment

This flowchart captures the full recommended process for the artisanal-vegetable A/B test.  
It emphasises **fixed-horizon** testing (run to the pre-calculated sample size) and the final step of **adapting the report to the audience** (data literacy, subject knowledge, and available time – see the supplied PDFs).

```mermaid
flowchart TD
    A[Start: Business Goal<br/>Increase weekly meal-plan revenue<br/>by ≥ $1 240] --> B[Gather Historical Data<br/>customer_visits, purchasing_customers,<br/>money_spent]
    B --> C[Compute Baseline Conversion Rate<br/>paying / total × 100]
    C --> D[Compute Revenue per Paying Customer<br/>np.mean(money_spent)]
    D --> E[New Customers Needed for +$1 240<br/>np.ceil(1240 / avg_payment)]
    E --> F[Percentage-Point Lift Required<br/>new_customers / total_visitors × 100]
    F --> G[Minimum Detectable Effect MDE<br/>relative % = pp_lift / baseline × 100]
    G --> H[Choose Significance Threshold<br/>α = 0.10 (10 %)]
    H --> I[Look up / Calculate Required Sample Size<br/>A/B Sample Size Calculator → ≈ 490]
    I --> J[Run Experiment to Fixed N<br/><b>Do NOT peek or stop early</b>]
    J --> K[Analyse Results<br/>conversion rates, lift, p-value,<br/>practical significance]
    K --> L{Statistically &amp; Practically<br/>Significant?}
    L -->|Yes| M[Adopt Artisanal Photos<br/>&amp; New Provider]
    L -->|No| N[Keep Current Assets<br/>or Iterate Hypothesis]
    M --> O[Write Data-Analysis Report<br/>Tailor to Audience:<br/>• Executives – headline + ROI<br/>• Technical – methods + code<br/>• Mixed – layered sections]
    N --> O
    O --> P[End: Update Baseline<br/>&amp; Share Organisational Learning]
    style J fill:#ffcccc,stroke:#cc0000
    style O fill:#e6f3ff,stroke:#0066cc
```

**Key takeaway (from the PDFs):** The last box is critical.  A C-level reader has little time and moderate data literacy; a data-science peer wants full methods and diagnostics.  Structure the report accordingly (Introduction → Body organised by question → Conclusion → Appendix).


## Audience Considerations (from the provided PDFs)

Before you write a single line of the final report, answer three questions about the people who will read it:

1. **Data Literacy**  
   - High (engineers, data scientists, psychology grads): scatter-plots, box-plots, confidence intervals are fine.  
   - Low (literature / fine-arts backgrounds): stick to bars, lines, simple percentages and friendly analogies.

2. **Subject Knowledge**  
   - Experts (finance, product, growth teams): skip definitions of EBITDA / WACC / conversion rate; they may prefer waterfall charts or profit-loss style layouts.  
   - Novices: explain every abbreviation and highlight whether an increase is good or bad.

3. **Time Span / Attention**  
   - C-level skimming a dashboard between meetings → one-sentence headline + recommendation.  
   - Same person reading a long-form magazine piece after hours → deeper narrative is acceptable.

The data-analysis report structure recommended in the third PDF is ideal for mixed audiences:

- **Introduction** – big questions + headline conclusions (executives stop here).  
- **Body** – one subsection per analytic question (methods → analysis → conclusion).  
- **Conclusion / Discussion** – reprise + next steps.  
- **Appendix** – full code, detailed tables, diagnostics (technical supervisors dig in here).


## 1. Setup – Import libraries


In [ ]:
import sys
sys.path.insert(0, "/home/workdir/artifacts")   # ensure mock library is found
import noshmishmosh
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")
print("noshmishmosh data available:")
print("  customer_visits length :", len(noshmishmosh.customer_visits))
print("  purchasing_customers   :", len(noshmishmosh.purchasing_customers))
print("  money_spent length     :", len(noshmishmosh.money_spent))


## 2. Baseline Conversion Rate (Steps 4-8)


In [ ]:
all_visitors = noshmishmosh.customer_visits
paying_visitors = noshmishmosh.purchasing_customers

total_visitor_count = len(all_visitors)
paying_visitor_count = len(paying_visitors)

baseline_percent = paying_visitor_count / total_visitor_count * 100.0
print("Baseline percent:")
print(baseline_percent)
print(f"→ {paying_visitor_count} paying out of {total_visitor_count} visitors")


## 3. Effect Size – Revenue Target → MDE (Steps 9-14)


In [ ]:
payment_history = noshmishmosh.money_spent
average_payment = np.mean(payment_history)
print(f"Average payment per paying customer: ${average_payment:.2f}")

new_customers_needed = np.ceil(1240 / average_payment)
print(f"New customers needed for +$1 240: {new_customers_needed:.0f}")

percentage_point_increase = new_customers_needed / total_visitor_count * 100
print("Percentage point increase:")
print(percentage_point_increase)

mde = percentage_point_increase / baseline_percent * 100.0
print("Minimum Detectable Effect (relative %):")
print(mde)


## 4. Significance Threshold & Final Sample Size (Steps 15-16)

Using the classic online A/B Sample Size Calculator with:
- Baseline ≈ 18.6 %
- MDE ≈ 50.5 % relative
- Significance = 10 %

the required total sample size is **490**.


In [ ]:
significance_threshold = 0.10
ab_sample_size = 490

print("Required sample size (total visitors to show the new assets):")
print(ab_sample_size)
print(f"≈ {ab_sample_size // 2} per variant (50/50 split)")


## 5. Alternate Code Paths (same results)


In [ ]:
# Alternate A – pure Python mean
average_payment_alt = sum(payment_history) / len(payment_history)
print("Alternate A – pure-Python mean:", round(average_payment_alt, 4))

# Alternate B – Boolean list comprehension for baseline
# (simulate a purchase flag list)
purchase_flags = [True] * paying_visitor_count + [False] * (total_visitor_count - paying_visitor_count)
baseline_alt = sum(1 for p in purchase_flags if p) / len(purchase_flags) * 100
print("Alternate B – list-comprehension baseline:", baseline_alt)

# Alternate C – recompute sample size with statsmodels
baseline_rate = baseline_percent / 100
target_rate = baseline_rate * (1 + mde / 100)
effect_size = proportion_effectsize(baseline_rate, target_rate)
analysis = NormalIndPower()
n_per_variant = analysis.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=significance_threshold,
    ratio=1.0
)
total_from_formula = int(np.ceil(n_per_variant * 2))
print(f"Alternate C – statsmodels sample size: {total_from_formula} (vs classic calculator 490)")


## 6. Extra Practice – Solutions


In [ ]:
print("=== Extra Practice 1: average payment rises to $30 ===")
new_needed_30 = np.ceil(1240 / 30)
pp_30 = new_needed_30 / total_visitor_count * 100
mde_30 = pp_30 / baseline_percent * 100
print(f"New customers needed: {new_needed_30:.0f}")
print(f"New MDE (relative %): {mde_30:.1f}")

print("\n=== Extra Practice 2: tighten α to 0.05 ===")
n_per_05 = analysis.solve_power(effect_size=effect_size, power=0.80, alpha=0.05, ratio=1.0)
print(f"Sample size at α=0.05: {int(np.ceil(n_per_05*2))}  (larger, as expected)")

print("\n=== Extra Practice 3: baseline only 10 % ===")
baseline_10 = 10.0
pp_same = percentage_point_increase          # same absolute revenue target → same pp lift
mde_10 = pp_same / baseline_10 * 100
print(f"New MDE at 10 % baseline: {mde_10:.1f} % relative")
es_10 = proportion_effectsize(0.10, 0.10 * (1 + mde_10/100))
n_10 = analysis.solve_power(effect_size=es_10, power=0.80, alpha=0.10, ratio=1.0)
print(f"Required sample size: {int(np.ceil(n_10*2))}")

print("\n=== Extra Practice 4: 15-second executive summary ===")
print("We need roughly 490 visitors to be confident (90 % significance) that the")
print("artisanal photos will lift weekly revenue by at least $1 240.  If the test")
print("succeeds we should switch providers; otherwise keep the current assets.")


## 7. Simulation Playground – Turn the Knobs

Change any of the four values below and re-run to see the impact on required sample size.


In [ ]:
# ========== KNOBS ==========
sim_baseline_pct = 18.6      # current conversion %
sim_mde_relative = 50.5      # relative lift we want to detect (%)
sim_alpha        = 0.10      # significance level
sim_power        = 0.80      # desired power
# ===========================

def required_sample_size(baseline_pct, mde_rel, alpha, power):
    p1 = baseline_pct / 100
    p2 = p1 * (1 + mde_rel / 100)
    es = proportion_effectsize(p1, p2)
    n_per = NormalIndPower().solve_power(effect_size=es, power=power, alpha=alpha, ratio=1.0)
    return int(np.ceil(n_per * 2))

base_n = required_sample_size(sim_baseline_pct, sim_mde_relative, sim_alpha, sim_power)
print(f"Current settings → total sample size = {base_n}")

# Sensitivity table
print("\nSensitivity to MDE (holding other knobs fixed):")
print(f"{'MDE %':>8} | {'Sample Size':>12}")
print("-" * 25)
for mde_try in [30, 40, 50.5, 60, 80]:
    n = required_sample_size(sim_baseline_pct, mde_try, sim_alpha, sim_power)
    print(f"{mde_try:>8.1f} | {n:>12}")

print("\nSensitivity to α:")
print(f"{'α':>8} | {'Sample Size':>12}")
print("-" * 25)
for a in [0.01, 0.05, 0.10, 0.20]:
    n = required_sample_size(sim_baseline_pct, sim_mde_relative, a, sim_power)
    print(f"{a:>8.2f} | {n:>12}")


## 8. Visualisation – How Sample Size Grows with Smaller Effects


In [ ]:
mdes = np.linspace(20, 100, 17)
sizes = [required_sample_size(18.6, m, 0.10, 0.80) for m in mdes]

plt.figure(figsize=(9, 5))
plt.plot(mdes, sizes, 'o-', color='#2a9d8f', linewidth=2, markersize=7)
plt.axvline(50.5, color='#e76f51', linestyle='--', label='Our MDE ≈ 50.5 %')
plt.axhline(490, color='#e76f51', linestyle=':', label='Our N ≈ 490')
plt.xlabel('Minimum Detectable Effect (relative %)')
plt.ylabel('Required Total Sample Size')
plt.title('Sample Size vs. Desired Lift (α=0.10, power=0.80, baseline=18.6 %)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Monte-Carlo Verification of Power

Does a sample of 490 really give us ~80 % power to detect a 50.5 % relative lift?


In [ ]:
from scipy.stats import chi2_contingency

def simulate_power(n_total, p_control, p_treatment, alpha=0.10, n_sim=3000):
    n_per = n_total // 2
    detections = 0
    for _ in range(n_sim):
        c = np.random.binomial(n_per, p_control)
        t = np.random.binomial(n_per, p_treatment)
        table = [[c, n_per - c], [t, n_per - t]]
        _, p, _, _ = chi2_contingency(table, correction=False)
        if p < alpha:
            detections += 1
    return detections / n_sim

p_base = baseline_percent / 100
p_treat = p_base * (1 + mde / 100)
emp_power = simulate_power(490, p_base, p_treat, alpha=0.10)
print(f"Empirical power at N=490, α=0.10: {emp_power*100:.1f} %")
print("(Should be close to the nominal 80 % – sampling variation is expected.)")


## 10. Audience-Adapted Communication

### C-level executive (30-second skim)
> “To be 90 % confident that the artisanal photos will add at least $1 240 of weekly revenue we must show the new images to roughly 490 visitors.  If the lift materialises we switch providers; otherwise we keep the current assets.”

### Data-science peer
> Full write-up of baseline (18.6 %), absolute revenue target → 47 extra customers → 9.4 pp → 50.5 % relative MDE, α = 0.10, power = 0.80, sample-size formula (or calculator) yielding 490, Monte-Carlo confirmation of power, and the stopping rule “run to fixed N, no peeking”.  Code and diagnostics live in the Appendix.

### Mixed product / design audience
> Layered report: one-sentence headline on page 1, a short “why 490?” explainer with the revenue-to-MDE chain, a simple bar chart of current vs target conversion, and a clear recommendation box.  Technical appendix available on request.


## 11. Summary of Key Results


In [ ]:
print("=" * 60)
print("NOSH MISH MOSH – A/B SAMPLE SIZE SUMMARY")
print("=" * 60)
print(f"Baseline conversion          : {baseline_percent:.1f} %")
print(f"Average payment              : ${average_payment:.2f}")
print(f"New customers needed (+$1240): {new_customers_needed:.0f}")
print(f"Percentage-point lift needed : {percentage_point_increase:.1f} pp")
print(f"Minimum Detectable Effect    : {mde:.1f} % relative")
print(f"Significance threshold (α)   : {significance_threshold}")
print(f"Required total sample size   : {ab_sample_size}")
print("=" * 60)
print("Remember: run the FULL sample size. Do not peek and stop early.")
print("Tailor the final report to the audience’s data literacy, subject")
print("knowledge and available time (see PDFs on audience analysis).")
